# Modelling Phase

In [1]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

# system
import os
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parents[2]))  # workaround to resolve my helper module above

# dataframe
import polars as pl
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# modules
from modules.plotting import *
from modules.polars_helpers import *

# stats
from scipy.stats import pearsonr

# plotting
from matplotlib.ticker import FuncFormatter

# NN
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, StringLookup, Embedding, Flatten, Concatenate
from keras.callbacks import EarlyStopping
from keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.data import Dataset

parent_dir = Path.cwd().parent

DATA_PATH = str(parent_dir / "data/")
print(DATA_PATH)

RANDOM_SEED = 42

2025-05-21 16:17:20.473458: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


/Users/luiscruz/Desktop/kaggle_challenges/regression/gcp-taxi-fare-prediction/data


In [2]:
train = pl.scan_parquet(DATA_PATH + "/final_eda_df.parquet")

test_csv = pl.read_csv(DATA_PATH + "/test.csv")
test_csv.schema

Schema([('key', String),
        ('pickup_datetime', String),
        ('pickup_longitude', Float64),
        ('pickup_latitude', Float64),
        ('dropoff_longitude', Float64),
        ('dropoff_latitude', Float64),
        ('passenger_count', Int64)])

In [3]:
test_csv.null_count()

key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


In [4]:
train.collect_schema()

Schema([('key', String),
        ('fare_amount', Float64),
        ('pickup_datetime', String),
        ('pickup_longitude', Float64),
        ('pickup_latitude', Float64),
        ('dropoff_longitude', Float64),
        ('dropoff_latitude', Float64),
        ('passenger_count', Int64),
        ('ride_distance', Float64),
        ('pickup_datetime_parsed', Datetime(time_unit='us', time_zone=None)),
        ('year', Int32),
        ('month', Int8),
        ('hour', Int8),
        ('day_of_week', Int8),
        ('pickup_nta', String),
        ('dropoff_nta', String),
        ('mean_fare_per_dropoff', Float64)])

In [5]:
train = train.drop("mean_fare_per_dropoff", "pickup_datetime_parsed", "pickup_datetime", "key")

In [6]:
test_csv.describe()

statistic,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
str,str,str,f64,f64,f64,f64,f64
"""count""","""9914""","""9914""",9914.0,9914.0,9914.0,9914.0,9914.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,-73.974722,40.751041,-73.973657,40.751743,1.671273
"""std""",null,null,0.042774,0.033541,0.039072,0.035435,1.278747
"""min""","""2009-01-01 11:04:24.0000001""","""2009-01-01 11:04:24 UTC""",-74.252193,40.573143,-74.263242,40.568973,1.0
"""25%""",null,null,-73.992502,40.736123,-73.991247,40.735254,1.0
"""50%""",null,null,-73.982323,40.753052,-73.980012,40.754067,1.0
"""75%""",null,null,-73.968013,40.767113,-73.964048,40.768758,2.0
"""max""","""2015-06-30 20:03:50.0000005""","""2015-06-30 20:03:50 UTC""",-72.986532,41.709555,-72.990963,41.696683,6.0


In [7]:
def clean(train: pl.LazyFrame):
    pass

In [8]:
def preprocess(test: pl.DataFrame) -> pl.DataFrame:

    # manhattan distance
    test = test.with_columns(
        (
            (pl.col("pickup_latitude") - pl.col("dropoff_latitude")).abs()
            +
            (pl.col("pickup_longitude") - pl.col("dropoff_longitude")).abs()
        ).alias("ride_distance")
    )

    # datetime
    test = test.with_columns([
        pl.col("pickup_datetime")
        .str.strip_chars()
        .str.strip_suffix(" UTC")
        .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
        .alias("pickup_datetime_parsed")
    ])

    test = test.with_columns([
        pl.col("pickup_datetime_parsed").dt.year().alias("year"),
        pl.col("pickup_datetime_parsed").dt.month().alias("month"),
        pl.col("pickup_datetime_parsed").dt.hour().alias("hour"),
        pl.col("pickup_datetime_parsed").dt.weekday().alias("day_of_week") # monday = 1 and sunday = 7
    ])

    # add nta data
    nta = gpd.read_file(DATA_PATH + "/nynta/nynta2020.shp").to_crs(epsg=4326)

    df = test.to_pandas()

    pickup_gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df.pickup_longitude, df.pickup_latitude)],
        crs="EPSG:4326"
    )
    dropoff_gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df.dropoff_longitude, df.dropoff_latitude)],
        crs="EPSG:4326"
    )

    pickup_joined = gpd.sjoin(pickup_gdf, nta, how="left", predicate="within")
    dropoff_joined = gpd.sjoin(dropoff_gdf, nta, how="left", predicate="within")

    df["pickup_nta"] = pickup_joined["NTAName"]
    df["dropoff_nta"] = dropoff_joined["NTAName"]

    test = pl.from_pandas(df).drop("pickup_datetime_parsed", "pickup_datetime", "key")

    return test

In [9]:
test_processed = preprocess(test_csv)
test_processed.schema

Schema([('pickup_longitude', Float64),
        ('pickup_latitude', Float64),
        ('dropoff_longitude', Float64),
        ('dropoff_latitude', Float64),
        ('passenger_count', Int64),
        ('ride_distance', Float64),
        ('year', Int32),
        ('month', Int8),
        ('hour', Int8),
        ('day_of_week', Int8),
        ('pickup_nta', String),
        ('dropoff_nta', String)])

In [10]:
train.collect_schema()

Schema([('fare_amount', Float64),
        ('pickup_longitude', Float64),
        ('pickup_latitude', Float64),
        ('dropoff_longitude', Float64),
        ('dropoff_latitude', Float64),
        ('passenger_count', Int64),
        ('ride_distance', Float64),
        ('year', Int32),
        ('month', Int8),
        ('hour', Int8),
        ('day_of_week', Int8),
        ('pickup_nta', String),
        ('dropoff_nta', String)])

In [11]:
train = train.collect().to_dummies(["passenger_count", "year", "month", "hour", "day_of_week"]).drop_nulls().sample(n=1_000_000, seed=RANDOM_SEED)
train

fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count_1,passenger_count_2,passenger_count_3,passenger_count_4,passenger_count_5,passenger_count_6,ride_distance,year_2009,year_2010,year_2011,year_2012,year_2013,year_2014,year_2015,month_1,month_10,month_11,month_12,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,hour_0,hour_1,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_2,hour_20,hour_21,hour_22,hour_23,hour_3,hour_4,hour_5,hour_6,hour_7,hour_8,hour_9,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6,day_of_week_7,pickup_nta,dropoff_nta
f64,f64,f64,f64,f64,u8,u8,u8,u8,u8,u8,f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,str,str
6.9,-73.991365,40.7506,-73.98197,40.757677,1,0,0,0,0,0,0.016472,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,"""Midtown South-Flatiron-Union S…","""Midtown-Times Square"""
9.5,-74.006859,40.730996,-73.99019,40.756819,1,0,0,0,0,0,0.042492,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,"""West Village""","""Hell's Kitchen"""
6.1,-73.975709,40.789371,-73.963866,40.807774,1,0,0,0,0,0,0.030246,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Upper West Side (Central)""","""Morningside Heights"""
10.5,-73.987181,40.720742,-73.94957,40.714118,1,0,0,0,0,0,0.044235,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Lower East Side""","""East Williamsburg"""
26.9,-74.008863,40.719917,-73.909979,40.76958,1,0,0,0,0,0,0.148547,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,"""Tribeca-Civic Center""","""Astoria (North)-Ditmars-Steinw…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8.5,-73.979125,40.7555,-73.99499,40.727677,0,0,0,0,1,0,0.043688,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,"""Midtown-Times Square""","""Greenwich Village"""
10.5,-73.982703,40.738623,-73.999185,40.724945,1,0,0,0,0,0,0.03016,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,"""Gramercy""","""SoHo-Little Italy-Hudson Squar…"
5.3,-73.968618,40.767368,-73.956265,40.784197,0,0,1,0,0,0,0.029182,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,"""Upper East Side-Carnegie Hill""","""Upper East Side-Carnegie Hill"""


## Baseline NN

In [12]:
train_shape = train.shape
train_shape

(1000000, 64)

In [13]:
def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def r_squared(y_true, y_pred):
    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - ss_res / (ss_tot + K.epsilon())

early_stop = EarlyStopping(
    monitor='val_rmse',
    patience=100,
    mode="min",
    restore_best_weights=True
)

In [14]:
pickup_ds = Dataset.from_tensor_slices(train.select("pickup_nta").unique().drop_nulls().to_series().to_list())
pickup_lookup = StringLookup(output_mode="int", oov_token="[UNK]")
pickup_lookup.adapt(pickup_ds)

dropoff_ds = Dataset.from_tensor_slices(train.select("dropoff_nta").unique().drop_nulls().to_series().to_list())
dropoff_lookup = StringLookup(output_mode="int", oov_token="[UNK]")
dropoff_lookup.adapt(dropoff_ds)

In [15]:
X = train.drop("fare_amount").to_pandas()
y = train["fare_amount"].to_pandas()

In [ ]:
pickup_data = X["pickup_nta"]
dropoff_data = X["dropoff_nta"]
X = X.drop(["pickup_nta", "dropoff_nta"], axis=1)

In [23]:
X.shape

(1000000, 61)

In [24]:
def make_base_model() -> Model:
    # Inputs (string)
    pickup_input = Input(shape=(1,), dtype="string", name="pickup_nta")
    dropoff_input = Input(shape=(1,), dtype="string", name="dropoff_nta")

    # String -> int
    pickup_idx = pickup_lookup(pickup_input)
    dropoff_idx = dropoff_lookup(dropoff_input)

    # Integer -> embedding
    pickup_emb = Embedding(input_dim=pickup_lookup.vocabulary_size(), output_dim=8)(pickup_idx)
    dropoff_emb = Embedding(input_dim=dropoff_lookup.vocabulary_size(), output_dim=8)(dropoff_idx)

    # Flatten to feed into model
    pickup_emb = Flatten()(pickup_emb)
    dropoff_emb = Flatten()(dropoff_emb)

    # Combine embeddings with other features
    numerical_input = Input(shape=(X.shape[1]), name="numerical_features")

    concat = Concatenate()([pickup_emb, dropoff_emb, numerical_input])

    x = Dense(128, activation="relu")(concat)
    x = Dropout(rate=0.2)(x)

    output = Dense(1, activation="relu")(x)

    model = Model(inputs=[pickup_emb, dropoff_emb, numerical_input], outputs=output)
    
    model.compile(optimizer="adam", loss="mse", metrics=[rmse, r_squared])

    return model

In [25]:
baseline_nn = make_base_model()
baseline_nn.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 8)]                  0         []                            
                                                                                                  
 input_4 (InputLayer)        [(None, 8)]                  0         []                            
                                                                                                  
 numerical_features (InputL  [(None, 61)]                 0         []                            
 ayer)                                                                                            
                                                                                                  
 concatenate_1 (Concatenate  (None, 77)                   0         ['input_3[0][0]',       

In [19]:
X

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count_1,passenger_count_2,passenger_count_3,passenger_count_4,passenger_count_5,passenger_count_6,...,hour_9,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,day_of_week_5,day_of_week_6,day_of_week_7,pickup_nta,dropoff_nta
0,-73.991365,40.750600,-73.981970,40.757677,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,Midtown South-Flatiron-Union Square,Midtown-Times Square
1,-74.006859,40.730996,-73.990190,40.756819,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,West Village,Hell's Kitchen
2,-73.975709,40.789371,-73.963866,40.807774,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,Upper West Side (Central),Morningside Heights
3,-73.987181,40.720742,-73.949570,40.714118,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,Lower East Side,East Williamsburg
4,-74.008863,40.719917,-73.909979,40.769580,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,Tribeca-Civic Center,Astoria (North)-Ditmars-Steinway
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,-73.979125,40.755500,-73.994990,40.727677,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,Midtown-Times Square,Greenwich Village
999996,-73.982703,40.738623,-73.999185,40.724945,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,Gramercy,SoHo-Little Italy-Hudson Square
999997,-73.968618,40.767368,-73.956265,40.784197,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,Upper East Side-Carnegie Hill,Upper East Side-Carnegie Hill
999998,-73.989551,40.748444,-73.985809,40.752316,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,Midtown South-Flatiron-Union Square,Midtown-Times Square


In [26]:
baseline_nn.fit(
    x={
        "input_3": pickup_data,
        "input_4": dropoff_data,
        "numerical_features": X,
    },
    y=y,
    validation_split=0.3,
    callbacks=[early_stop],
    batch_size=100_000,
    epochs=250,
    verbose=2
)

Epoch 1/250


ValueError: in user code:

    File "/Users/luiscruz/Desktop/kaggle_challenges/.venv/lib/python3.11/site-packages/keras/src/engine/training.py", line 1401, in train_function  *
        return step_function(self, iterator)
    File "/Users/luiscruz/Desktop/kaggle_challenges/.venv/lib/python3.11/site-packages/keras/src/engine/training.py", line 1384, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/Users/luiscruz/Desktop/kaggle_challenges/.venv/lib/python3.11/site-packages/keras/src/engine/training.py", line 1373, in run_step  **
        outputs = model.train_step(data)
    File "/Users/luiscruz/Desktop/kaggle_challenges/.venv/lib/python3.11/site-packages/keras/src/engine/training.py", line 1150, in train_step
        y_pred = self(x, training=True)
    File "/Users/luiscruz/Desktop/kaggle_challenges/.venv/lib/python3.11/site-packages/keras/src/utils/traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filter